In [14]:
import pandas as pd
from IPython.display import display, HTML

In [15]:
all_results = pd.read_csv("all_results.csv",)

In [16]:
all_results

,dataset,model,run,training_time_s,sampling_time_s,cls_Real vs Real (baseline)_ROC-AUC,cls_Real vs Real (baseline)_Precision,cls_Real vs Real (baseline)_Recall,cls_Real vs Real (baseline)_F1,cls_Gen vs Real (TSTR)_ROC-AUC,...,qual_Val vs Gen_Coverage,spd_Real train_Sym.,spd_Real train_Pos. def.,spd_Real train_in M,spd_Generated train_Sym.,spd_Generated train_Pos. def.,spd_Generated train_in M,spd_Generated val_Sym.,spd_Generated val_Pos. def.,spd_Generated val_in M
0,BNCI2014002,diffeocfm,group_0,2522.4,0.8,0.8352,0.7179,0.7467,0.7320,0.7662,...,1.0000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,BNCI2014002,diffeocfm,group_1,1986.4,0.6,0.8097,0.6667,0.7941,0.7248,0.7532,...,0.9671,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,BNCI2014002,diffeocfm,group_2,2642.0,0.5,0.8318,0.6988,0.8056,0.7484,0.8082,...,0.9934,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,BNCI2014002,diffeocfm,group_3,2159.9,0.4,0.7964,0.7237,0.7432,0.7333,0.8032,...,0.9934,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,BNCI2014002,diffeocfm,group_4,2797.9,0.4,0.8315,0.7206,0.6901,0.7050,0.8294,...,0.9934,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
5,BNCI2014002,geodesiccfm,group_0,1807.7,1.4,0.8352,0.7179,0.7467,0.7320,0.8023,...,1.0000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
6,BNCI2014002,geodesiccfm,group_1,1313.2,1.3,0.8097,0.6667,0.7941,0.7248,0.7987,...,1.0000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
7,BNCI2014002,geodesiccfm,group_2,1811.7,1.3,0.8318,0.6988,0.8056,0.7484,0.7920,...,1.0000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
8,BNCI2014002,geodesiccfm,group_3,1311.1,1.4,0.7964,0.7237,0.7432,0.7333,0.8105,...,1.0000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
9,BNCI2014002,geodesiccfm,group_4,1813.8,1.3,0.8315,0.7206,0.6901,0.7050,0.8262,...,1.0000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [17]:
MODEL_ORDER  = ["diffeocfm", "geodesiccfm", "wrappedcfm", "wrappedcfm_with_prior"]
MODEL_LABELS = {"diffeocfm": "DiffeoCFM", "geodesiccfm": "GeodesicCFM",
                "wrappedcfm": "WrappedCFM", "wrappedcfm_with_prior": "WrappedCFM (prior)"}
DATASETS = sorted(all_results["dataset"].unique())

def make_table(df, metrics: dict, baseline_cols: dict = None) -> pd.DataFrame:
    """Build a mean±std table: rows=models, columns=metrics, one per dataset."""
    frames = {}
    for dataset in DATASETS:
        ds = df[df["dataset"] == dataset]
        rows = {}
        for model in MODEL_ORDER:
            m = ds[ds["model"] == model]
            row = {}
            for label, col in metrics.items():
                mean, std = m[col].mean(), m[col].std()
                row[label] = f"{mean:.3f} ± {std:.3f}"
            rows[MODEL_LABELS[model]] = row
        tbl = pd.DataFrame(rows).T
        if baseline_cols:
            baseline_row = {}
            for label, col in baseline_cols.items():
                val = ds[col].mean()
                baseline_row[label] = f"{val:.3f}"
            tbl.loc["— baseline —"] = baseline_row
        frames[dataset] = tbl
    return frames

## Classification metrics (TSTR & TRTS)

In [18]:
cls_metrics = {
    "TSTR ROC-AUC": "cls_Gen vs Real (TSTR)_ROC-AUC",
    "TSTR F1":      "cls_Gen vs Real (TSTR)_F1",
    "TRTS ROC-AUC": "cls_Real vs Gen (TRTS)_ROC-AUC",
    "TRTS F1":      "cls_Real vs Gen (TRTS)_F1",
}
cls_baselines = {
    "TSTR ROC-AUC": "cls_Real vs Real (baseline)_ROC-AUC",
    "TSTR F1":      "cls_Real vs Real (baseline)_F1",
    "TRTS ROC-AUC": "cls_Real vs Real (baseline)_ROC-AUC",
    "TRTS F1":      "cls_Real vs Real (baseline)_F1",
}

tables = make_table(all_results, cls_metrics, cls_baselines)
for dataset, tbl in tables.items():
    display(HTML(f"<h3>{dataset}</h3>"))
    display(tbl)

,TSTR ROC-AUC,TSTR F1,TRTS ROC-AUC,TRTS F1
DiffeoCFM,0.792 ± 0.031,0.685 ± 0.050,0.874 ± 0.029,0.766 ± 0.040
GeodesicCFM,0.806 ± 0.013,0.706 ± 0.017,0.892 ± 0.023,0.791 ± 0.019
WrappedCFM,0.811 ± 0.014,0.715 ± 0.020,0.855 ± 0.012,0.764 ± 0.024
WrappedCFM (prior),0.798 ± 0.017,0.718 ± 0.032,0.797 ± 0.040,0.687 ± 0.058
— baseline —,0.821,0.729,0.821,0.729


,TSTR ROC-AUC,TSTR F1,TRTS ROC-AUC,TRTS F1
DiffeoCFM,0.818 ± 0.002,0.729 ± 0.023,0.898 ± 0.013,0.816 ± 0.006
GeodesicCFM,0.816 ± 0.016,0.718 ± 0.008,0.889 ± 0.033,0.816 ± 0.013
WrappedCFM,0.796 ± 0.001,0.724 ± 0.008,0.831 ± 0.009,0.740 ± 0.012
WrappedCFM (prior),0.793 ± 0.028,0.701 ± 0.012,0.802 ± 0.031,0.704 ± 0.039
— baseline —,0.807,0.727,0.807,0.727


## Quality metrics (Train vs Gen)

In [19]:
qual_metrics = {
    "Precision": "qual_Train vs Gen_Precision",
    "Recall":    "qual_Train vs Gen_Recall",
    "Density":   "qual_Train vs Gen_Density",
    "Coverage":  "qual_Train vs Gen_Coverage",
}
qual_baselines = {
    "Precision": "qual_Train vs Val (baseline)_Precision",
    "Recall":    "qual_Train vs Val (baseline)_Recall",
    "Density":   "qual_Train vs Val (baseline)_Density",
    "Coverage":  "qual_Train vs Val (baseline)_Coverage",
}

tables = make_table(all_results, qual_metrics, qual_baselines)
for dataset, tbl in tables.items():
    display(HTML(f"<h3>{dataset}</h3>"))
    display(tbl)

,Precision,Recall,Density,Coverage
DiffeoCFM,0.496 ± 0.032,0.986 ± 0.001,0.276 ± 0.040,0.760 ± 0.036
GeodesicCFM,0.714 ± 0.028,0.975 ± 0.002,0.576 ± 0.081,0.942 ± 0.020
WrappedCFM,0.718 ± 0.016,0.972 ± 0.003,0.607 ± 0.077,0.939 ± 0.012
WrappedCFM (prior),0.710 ± 0.035,0.973 ± 0.006,0.554 ± 0.080,0.924 ± 0.018
— baseline —,0.993,0.992,0.945,0.951


,Precision,Recall,Density,Coverage
DiffeoCFM,0.645 ± 0.002,0.956 ± 0.007,0.335 ± 0.027,0.789 ± 0.004
GeodesicCFM,0.884 ± 0.006,0.931 ± 0.009,0.934 ± 0.011,0.989 ± 0.001
WrappedCFM,0.909 ± 0.023,0.915 ± 0.001,1.072 ± 0.095,0.992 ± 0.004
WrappedCFM (prior),0.891 ± 0.002,0.922 ± 0.002,0.974 ± 0.032,0.978 ± 0.010
— baseline —,0.979,0.993,1.004,0.998


## Training & sampling time

In [20]:
time_metrics = {
    "Training time (s)": "training_time_s",
    "Sampling time (s)": "sampling_time_s",
}

tables = make_table(all_results, time_metrics)
for dataset, tbl in tables.items():
    display(HTML(f"<h3>{dataset}</h3>"))
    display(tbl)

,Training time (s),Sampling time (s)
DiffeoCFM,2421.720 ± 338.461,0.540 ± 0.167
GeodesicCFM,1611.500 ± 273.278,1.340 ± 0.055
WrappedCFM,2354.740 ± 327.387,2.120 ± 0.466
WrappedCFM (prior),2874.480 ± 502.205,1.440 ± 0.114


,Training time (s),Sampling time (s)
DiffeoCFM,746.350 ± 5.869,0.450 ± 0.071
GeodesicCFM,1805.100 ± 28.143,3.850 ± 2.616
WrappedCFM,1688.150 ± 42.921,1.750 ± 0.071
WrappedCFM (prior),2402.700 ± 30.406,1.500 ± 0.000
